# Equity champions — feature importance (train-only, clean split)

Champions from `selection_table.csv` — purged inner k-fold + 1SE, global cut **2021-10-06**.
All importance analysis uses **train data only** (post-split). The held-out 30% (post-2021-10-20) is **sealed**.

| Instrument | Group | Champion | AUC | Lower CI | Signal |
|------------|-------|----------|-----|----------|--------|
| es1s | es1s | RF | 0.516±0.113 | 0.40 | NO |
| nq1s | nq1s | XGB | 0.689±0.073 | 0.62 | YES |
| fesx1s | fesx1s | Logistic | 0.579±0.060 | 0.52 | YES |

> **es1s** carries no confirmed signal (lower CI < 0.50). Its importance analysis is included for contrast only — results reflect model noise rather than exploitable structure.


---
## Analysis pipeline

The same five-step sequence is applied to each instrument. All computations run on **train data only**;
the sealed 30% test set is never read here and no model is evaluated OOS.

### Step 1 — Feature clustering
Continuous features (excluding hand-assigned groups) are clustered by **Spearman distance** √(1−|ρ|)
with **Ward linkage**. The number of clusters K is chosen by silhouette score over K ∈ [2, 25].
Four groups are hand-assigned by prefix: `f4_` → F4_latent, `f5_` → F5_signal, `f8_` → F8_calendar,
`inst_` → F_instrument (pooled models only). These bypass correlation clustering.

### Step 2 — Cluster-level importance
Three importance signals per cluster, evaluated within **CPCV** (n_groups=6, k=2, embargo=1 %):

- **Clustered MDA** (headline): jointly permute all cluster members with one shared row permutation;
  N=10 repeats averaged; score AUC drop on target instrument's CPCV test slice.
  For pooled models the test slice is filtered to the target instrument before scoring.
- **Tree models** — MDI (train; sum feature_importances_ within cluster) and Group SHAP
  (TreeSHAP, tree_path_dependent, OOS slice).
- **Logistic model** — cluster |coef| sum (sum of |standardised elastic-net coefficients|
  per cluster). MDA permutes the **scaled** data (scaler fitted on outer-train fold).

A cluster is **significant** when its mean MDA drop > 1σ (cross-CPCV std).
**Rank agreement** is assessed via Kendall τ across cluster MDA / MDI / SHAP rankings
(or MDA / Coef for logistic); |τ| > 0.4 is treated as agreement.

### Step 3 — Within-cluster breakdown (top 3 clusters)
For the top 3 clusters by MDA (preferring significant clusters), individual members are ranked
by mean |SHAP| (tree) or mean |coef| (logistic). Single-feature permutation is intentionally
**not used** within a cluster — correlated members would inflate individual scores.

### Step 4 — PCA within top clusters
**PCA** (up to 3 PCs, standardised inputs) on the cluster's train submatrix.
PC1 variance interpretation tier: ≥65 % = single latent dimension; 40–65 % = dominant direction;
<40 % = genuinely multi-dimensional. Loadings reveal which members drive each component.

### Step 5 — Global per-feature view
**Tree**: mean |SHAP| per feature (averaged across CPCV paths), coloured by sign (red = positive,
blue = negative predictive direction). Also reports per-feature MDI.
**Logistic**: mean |standardised coefficient| per feature, coloured by sign.
SHAP direction reflects **model direction**, not necessarily profitable direction — the model may
learn to trade against a feature.


In [ ]:
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from IPython.display import Image, display

BASE = Path('outputs/importance')

def show(path, width=1100):
    p = Path(path)
    if p.exists():
        display(Image(str(p), width=width))
    else:
        print(f'[missing] {p}')

def load(path):
    p = Path(path)
    if p.exists():
        return pd.read_csv(p)
    print(f'[missing] {p}')
    return pd.DataFrame()

def cluster_summary(inst):
    mem = load(BASE / inst / 'cluster_membership.csv')
    mda = load(BASE / inst / 'clustered_mda_full.csv')
    if mem.empty or mda.empty:
        return pd.DataFrame()
    grp = (
        mem.groupby('cluster')
        .agg(
            n_members=('feature', 'count'),
            dominant_pfx=('f_prefix', lambda x: x.value_counts().index[0]),
            purity=('f_prefix', lambda x: round(x.value_counts().iloc[0] / len(x), 2)),
        )
        .reset_index()
    )
    mda_sub = mda[['cluster', 'mean_drop', 'std_drop', 'significant']].copy()
    return grp.merge(mda_sub, on='cluster', how='left').sort_values('mean_drop', ascending=False)

print('Setup complete. Artifact root:', BASE.resolve())


---
## ES1S — champion: `es1s` / RF

**CPCV:** 15 paths · AUC 0.548 ± 0.112 (held-out CPCV) · **NO SIGNAL** (lower CI 0.40)

**Significant clusters:** F5\_signal only.

> **Interpretation caveat:** es1s has no confirmed signal. The dominant cluster (F5_signal, MDA 0.083)
> reflects the model's reliance on internal signal-quality features rather than external market
> predictors. Weak MDA–MDI (τ=0.18) and MDA–SHAP (τ=0.24) agreement with strong MDI–SHAP agreement
> (τ=0.91) suggests the tree's internal metrics are coherent but MDA identifies different,
> noisier structure. Results are included for contrast only.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('es1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.02, vmax=0.10)
    .set_caption('es1s — cluster summary (K=14 corr + 3 hand-assigned = 17 groups)')
)
show(BASE / 'es1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

Clustered MDA (dark = significant > 1σ, light = inconclusive). Cross-check with MDI and SHAP.
Rank agreement assessed via Kendall τ; |τ| > 0.4 = agree.


In [ ]:
show(BASE / 'es1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'es1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'es1s' / 'rank_agreement.csv')
if not cc.empty:
    display(
        cc.style
        .format({'mda_mean': '{:.4f}', 'mdi_sum': '{:.4f}', 'shap_sum': '{:.4f}'})
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.05, vmax=0.10)
        .set_caption('es1s — cluster cross-check (MDA · MDI · SHAP ranks)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement:')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (top 3 by MDA)

Members ranked by mean |SHAP| (tree_path_dependent, OOS). PCA on train submatrix.

**F5_signal** (signal-quality features) is the only significant cluster — meaning the RF
gains primarily by learning the regime/participation/trailing-run context of its own signals,
not from external market predictors.


In [ ]:
wc = load(BASE / 'es1s' / 'within_cluster_F5_signal.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2 = wc['pca_pc2_var_explained'].iloc[0] if 'pca_pc2_var_explained' in wc.columns else float('nan')
    tier = '>= 65% => single latent dim' if pc1 >= 0.65 else ('40-65% => dominant direction' if pc1 >= 0.40 else '< 40% => multi-dimensional')
    label = f'PC1={pc1:.1%} PC2={pc2:.1%} ({tier})'
    display(
        wc[['feature', 'mean_shap_mag', 'pc1_loading']].style
        .format({'mean_shap_mag': '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=['mean_shap_mag'], cmap='Blues')
        .set_caption(f'es1s | F5_signal — SHAP ranking + PC1 loadings  ({label})')
    )
show(BASE / 'es1s' / 'within_cluster_F5_signal.png', width=950)


In [ ]:
wc = load(BASE / 'es1s' / 'within_cluster_C14_f11.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2 = wc['pca_pc2_var_explained'].iloc[0] if 'pca_pc2_var_explained' in wc.columns else float('nan')
    tier = '>= 65% => single latent dim' if pc1 >= 0.65 else ('40-65% => dominant direction' if pc1 >= 0.40 else '< 40% => multi-dimensional')
    label = f'PC1={pc1:.1%} PC2={pc2:.1%} ({tier})'
    display(
        wc[['feature', 'mean_shap_mag', 'pc1_loading']].style
        .format({'mean_shap_mag': '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=['mean_shap_mag'], cmap='Blues')
        .set_caption(f'es1s | C14_f11 — SHAP ranking + PC1 loadings  ({label})')
    )
show(BASE / 'es1s' / 'within_cluster_C14_f11.png', width=950)


In [ ]:
wc = load(BASE / 'es1s' / 'within_cluster_C2_f2.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2 = wc['pca_pc2_var_explained'].iloc[0] if 'pca_pc2_var_explained' in wc.columns else float('nan')
    tier = '>= 65% => single latent dim' if pc1 >= 0.65 else ('40-65% => dominant direction' if pc1 >= 0.40 else '< 40% => multi-dimensional')
    label = f'PC1={pc1:.1%} PC2={pc2:.1%} ({tier})'
    display(
        wc[['feature', 'mean_shap_mag', 'pc1_loading']].style
        .format({'mean_shap_mag': '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=['mean_shap_mag'], cmap='Blues')
        .set_caption(f'es1s | C2_f2 — SHAP ranking + PC1 loadings  ({label})')
    )
show(BASE / 'es1s' / 'within_cluster_C2_f2.png', width=950)


### Step 5 · Global per-feature SHAP

Mean |SHAP| per feature, averaged across 15 CPCV paths. Red = positive predictive direction,
blue = negative. Note: results for es1s reflect model noise rather than confirmed signal structure.


In [ ]:
gs = load(BASE / 'es1s' / 'global_shap_summary.csv')
if not gs.empty:
    display(
        gs.head(25).style
        .format({'shap_magnitude': '{:.4f}', 'shap_signed': '{:.4f}', 'mdi': '{:.4f}'})
        .background_gradient(subset=['shap_magnitude'], cmap='Blues')
        .set_caption('es1s — top 25 features by mean|SHAP| (NO SIGNAL — illustrative only)')
    )
show(BASE / 'es1s' / 'global_shap_chart.png', width=1000)


---
## NQ1S — champion: `nq1s` / XGB

**CPCV:** 15 paths · AUC 0.720 ± 0.073 · **SIGNAL** (lower CI 0.62)

**Significant clusters:** F5\_signal (MDA 0.174 ± 0.074), C4\_f11 (MDA 0.063 ± 0.050).

> NQ1S is the strongest equity champion. Two clusters clear the 1σ bar. The dominant driver
> `f11_dist_stock_surprise` (distillate stock surprise) in C4_f11 is striking for an equity index —
> it likely captures risk-off macro regime via energy inventory shocks. MDA–SHAP agree (τ=0.49),
> indicating the headline importance ranking is robust.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('nq1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.02, vmax=0.20)
    .set_caption('nq1s — cluster summary (K=15 corr + 3 hand-assigned = 18 groups)')
)
show(BASE / 'nq1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

Clustered MDA (dark = significant > 1σ). Cross-check with MDI and SHAP.


In [ ]:
show(BASE / 'nq1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'nq1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'nq1s' / 'rank_agreement.csv')
if not cc.empty:
    display(
        cc.style
        .format({'mda_mean': '{:.4f}', 'mdi_sum': '{:.4f}', 'shap_sum': '{:.4f}'})
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.05, vmax=0.20)
        .set_caption('nq1s — cluster cross-check (MDA · MDI · SHAP ranks)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement:')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (top 3 by MDA)

Members ranked by mean |SHAP|. PCA on train submatrix.

**F5_signal** PC1 explains 38% of variance — genuinely multi-dimensional (4 members span diverse
signal-quality axes). `f5_signal` dominates by SHAP (0.298, >> others).

**C4_f11** (4 mixed-f-prefix members): `f11_dist_stock_surprise` dominates (SHAP 0.061),
PC1 explains 67% suggesting one latent macro risk dimension.


In [ ]:
wc = load(BASE / 'nq1s' / 'within_cluster_F5_signal.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2 = wc['pca_pc2_var_explained'].iloc[0] if 'pca_pc2_var_explained' in wc.columns else float('nan')
    tier = '>= 65% => single latent dim' if pc1 >= 0.65 else ('40-65% => dominant direction' if pc1 >= 0.40 else '< 40% => multi-dimensional')
    label = f'PC1={pc1:.1%} PC2={pc2:.1%} ({tier})'
    display(
        wc[['feature', 'mean_shap_mag', 'pc1_loading']].style
        .format({'mean_shap_mag': '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=['mean_shap_mag'], cmap='Blues')
        .set_caption(f'nq1s | F5_signal — SHAP ranking + PC1 loadings  ({label})')
    )
show(BASE / 'nq1s' / 'within_cluster_F5_signal.png', width=950)


In [ ]:
wc = load(BASE / 'nq1s' / 'within_cluster_C4_f11.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2 = wc['pca_pc2_var_explained'].iloc[0] if 'pca_pc2_var_explained' in wc.columns else float('nan')
    tier = '>= 65% => single latent dim' if pc1 >= 0.65 else ('40-65% => dominant direction' if pc1 >= 0.40 else '< 40% => multi-dimensional')
    label = f'PC1={pc1:.1%} PC2={pc2:.1%} ({tier})'
    display(
        wc[['feature', 'mean_shap_mag', 'pc1_loading']].style
        .format({'mean_shap_mag': '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=['mean_shap_mag'], cmap='Blues')
        .set_caption(f'nq1s | C4_f11 — SHAP ranking + PC1 loadings  ({label})')
    )
show(BASE / 'nq1s' / 'within_cluster_C4_f11.png', width=950)


In [ ]:
wc = load(BASE / 'nq1s' / 'within_cluster_C8_f11.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2 = wc['pca_pc2_var_explained'].iloc[0] if 'pca_pc2_var_explained' in wc.columns else float('nan')
    tier = '>= 65% => single latent dim' if pc1 >= 0.65 else ('40-65% => dominant direction' if pc1 >= 0.40 else '< 40% => multi-dimensional')
    label = f'PC1={pc1:.1%} PC2={pc2:.1%} ({tier})'
    display(
        wc[['feature', 'mean_shap_mag', 'pc1_loading']].style
        .format({'mean_shap_mag': '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=['mean_shap_mag'], cmap='Blues')
        .set_caption(f'nq1s | C8_f11 — SHAP ranking + PC1 loadings  ({label})')
    )
show(BASE / 'nq1s' / 'within_cluster_C8_f11.png', width=950)


### Step 5 · Global per-feature SHAP

Mean |SHAP| per feature, averaged across 15 CPCV paths on nq1s. Red = positive, blue = negative.


In [ ]:
gs = load(BASE / 'nq1s' / 'global_shap_summary.csv')
if not gs.empty:
    display(
        gs.head(25).style
        .format({'shap_magnitude': '{:.4f}', 'shap_signed': '{:.4f}', 'mdi': '{:.4f}'})
        .background_gradient(subset=['shap_magnitude'], cmap='Blues')
        .set_caption('nq1s — top 25 features by mean|SHAP|')
    )
show(BASE / 'nq1s' / 'global_shap_chart.png', width=1000)


---
## FESX1S — champion: `fesx1s` / Logistic (elastic-net)

**CPCV:** 15 paths · AUC 0.582 ± 0.060 · **SIGNAL** (lower CI 0.52)

**Significant clusters:** F5\_signal only (MDA 0.065 ± 0.041). MDA–Coef agreement τ=0.45 (agree).

> fesx1s uses an **elastic-net logistic** champion. Importance analysis therefore uses:
> **clustered MDA** (model-agnostic, permutes scaled data) as the headline; and
> **sum of |standardised coefficients|** per cluster as the cross-check (replaces MDI/SHAP).
> No SHAP or MDI is available for this family.
>  
> Within C10_f12, `f12_mra_energy_D1` (multi-resolution analysis energy wavelet, scale 1)
> carries a coefficient of 0.242 — far above the other 3 members — suggesting a specific
> short-frequency energy signal drives Euro Stoxx positioning.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('fesx1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.02, vmax=0.08)
    .set_caption('fesx1s — cluster summary (K=15 corr + 3 hand-assigned = 18 groups)')
)
show(BASE / 'fesx1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

Clustered MDA (dark = significant > 1σ). Cross-check with elastic-net |coef| sums (no MDI/SHAP for logistic).


In [ ]:
show(BASE / 'fesx1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'fesx1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'fesx1s' / 'rank_agreement.csv')
if not cc.empty:
    fmt_cols = {'mda_mean': '{:.4f}'}
    if 'coef_sum' in cc.columns:
        fmt_cols['coef_sum'] = '{:.4f}'
    display(
        cc.style
        .format(fmt_cols)
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.05, vmax=0.08)
        .set_caption('fesx1s — cluster cross-check (MDA · Coef ranks; elastic-net logistic)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement (MDA vs Coef):')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (top 3 by MDA)

Members ranked by mean |standardised coef|. PCA on train submatrix.

**F5_signal** PC1 explains 36% — multi-dimensional; `f5_signal` (0.335) and `f5_trailing_run_length` (0.185) both contribute.

**C10_f12** PC1 explains 64% (dominant direction): `f12_mra_energy_D1` loads strongly
negative (−0.535) while `f2_ret_kurt_60` and `f12_mra_energy_D4/D5` load positive —
one wavelet-energy risk dimension drives the cluster.

**C13_f7** (OI features): PC1 explains only 44% — open-interest change, z-score, and
price-divergence form partially independent axes, though `f7_oi_z_20` dominates by coefficient.


In [ ]:
wc = load(BASE / 'fesx1s' / 'within_cluster_F5_signal.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2 = wc['pca_pc2_var_explained'].iloc[0] if 'pca_pc2_var_explained' in wc.columns else float('nan')
    tier = '>= 65% => single latent dim' if pc1 >= 0.65 else ('40-65% => dominant direction' if pc1 >= 0.40 else '< 40% => multi-dimensional')
    label = f'PC1={pc1:.1%} PC2={pc2:.1%} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else 'mean_shap_mag'
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'fesx1s | F5_signal — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'fesx1s' / 'within_cluster_F5_signal.png', width=950)


In [ ]:
wc = load(BASE / 'fesx1s' / 'within_cluster_C10_f12.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2 = wc['pca_pc2_var_explained'].iloc[0] if 'pca_pc2_var_explained' in wc.columns else float('nan')
    tier = '>= 65% => single latent dim' if pc1 >= 0.65 else ('40-65% => dominant direction' if pc1 >= 0.40 else '< 40% => multi-dimensional')
    label = f'PC1={pc1:.1%} PC2={pc2:.1%} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else 'mean_shap_mag'
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'fesx1s | C10_f12 — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'fesx1s' / 'within_cluster_C10_f12.png', width=950)


In [ ]:
wc = load(BASE / 'fesx1s' / 'within_cluster_C13_f7.csv')
if not wc.empty:
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2 = wc['pca_pc2_var_explained'].iloc[0] if 'pca_pc2_var_explained' in wc.columns else float('nan')
    tier = '>= 65% => single latent dim' if pc1 >= 0.65 else ('40-65% => dominant direction' if pc1 >= 0.40 else '< 40% => multi-dimensional')
    label = f'PC1={pc1:.1%} PC2={pc2:.1%} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else 'mean_shap_mag'
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'fesx1s | C13_f7 — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'fesx1s' / 'within_cluster_C13_f7.png', width=950)


### Step 5 · Global per-feature coefficient

Mean |standardised elastic-net coefficient| per feature, averaged across 15 CPCV paths.
Red = positive predictive direction (higher feature → higher P(bin=1)), blue = negative.


In [ ]:
gc = load(BASE / 'fesx1s' / 'global_coef_summary.csv')
if not gc.empty:
    display(
        gc.head(25).style
        .format({'coef_abs': '{:.4f}', 'coef_signed': '{:.4f}'})
        .background_gradient(subset=['coef_abs'], cmap='Blues')
        .set_caption('fesx1s — top 25 features by mean |standardised coefficient|')
    )
show(BASE / 'fesx1s' / 'global_coef_chart.png', width=1000)


---
## Cross-instrument findings — equity

Structured summary across es1s, nq1s, fesx1s.


In [ ]:
EQUITY_INSTS = ['es1s', 'nq1s', 'fesx1s']
NOSIGNAL = {'es1s'}

rows = []
for inst in EQUITY_INSTS:
    mda = load(BASE / inst / 'clustered_mda_full.csv')
    ra  = load(BASE / inst / 'rank_agreement.csv')
    meta = load(BASE / inst / 'champion_meta.csv')
    if mda.empty:
        continue
    top = mda.iloc[0]
    sig_count = int(mda['significant'].sum())
    tau_1 = ra['kendall_tau'].iloc[0] if not ra.empty else float('nan')
    champion = meta['model_type'].iloc[0].upper() if not meta.empty else '?'
    signal = meta['signal'].iloc[0] if not meta.empty else False
    rows.append({
        'inst': inst,
        'champion': champion,
        'signal': '\u2713' if signal else '\u2717',
        'n_sig_clusters': sig_count,
        'top_cluster': top['cluster'],
        'top_mda': round(top['mean_drop'], 4),
        'tau_primary': round(tau_1, 2) if not pd.isna(tau_1) else 'n/a',
    })

summary = pd.DataFrame(rows).set_index('inst')
display(
    summary.style
    .set_caption('Equity champions — cross-instrument summary')
)


In [ ]:
# Top cluster per instrument — MDA ranking
for inst in EQUITY_INSTS:
    mda = load(BASE / inst / 'clustered_mda_full.csv')
    if mda.empty:
        continue
    label = '(NO SIGNAL)' if inst in NOSIGNAL else ''
    print(f'\n{inst.upper()} {label}')
    print(mda[['cluster', 'mean_drop', 'std_drop', 'significant']].head(5).to_string(index=False))


### Key observations

**F5_signal cluster dominates all three equity champions**, suggesting that signal-quality
meta-features (trailing run length, participation rate, long bias) are the primary driver
of equity predictability — across RF, XGB, and logistic families.

**nq1s** is the only instrument with a second confirmed cluster (C4\_f11): mixed macro/energy
features including `f11_dist_stock_surprise`. The SHAP rank agrees with MDA (τ=0.49), lending
credibility to this as a real market structure feature for Nasdaq.

**fesx1s** (Euro Stoxx) shows `f12_mra_energy_D1` (energy wavelet scale-1) as the second-most
important feature by coefficient, hinting at a transatlantic energy-risk linkage for European
equity futures not seen in US indices.

**es1s** results should not be over-interpreted — the signal is not confirmed and the analysis
is included for structural contrast only.
